In [ ]:
!pip install langchain_huggingface langchain_text_splitters

In [ ]:
from langchain_huggingface import HuggingFaceEmbeddings
from typing import List
from langchain_text_splitters import HTMLSemanticPreservingSplitter
import zipfile
import os
import json
import pickle
from tqdm import tqdm

In [ ]:
SOURCE_ZIP_PATH = 'source.zip'
SOURCE_PATH = 'data/raw/QA'

with zipfile.ZipFile(SOURCE_ZIP_PATH, 'r') as zip_ref:
    zip_ref.extractall(SOURCE_PATH)

In [ ]:
class ConfluenceChunkEmbedder:
    def __init__(self, embedding_model_path = 'BAAI/bge-m3'):
        self.embedding_model = HuggingFaceEmbeddings(
            model_name = embedding_model_path,
            model_kwargs={'device': 'cuda'},
            encode_kwargs={
                "normalize_embeddings": True,
                'batch_size': 8
            }
        )

    def embed_chunks(self, chunks: List):
        texts = [chunk.page_content for chunk in chunks]
        return self.embedding_model.embed_documents(texts)

class ConfluenceChunker:
    def __init__(self):
        headers_to_split_on = [
            ("h2", "header 2"),
            ('tr', 'table_row')
        ]
        self.text_splitter = HTMLSemanticPreservingSplitter(
            headers_to_split_on=headers_to_split_on,
            elements_to_preserve=["table", "ul", "ol", "code"]
        )

    def basic_chunk(self, html_content):
        chunks = self.text_splitter.split_text(html_content)

        no_duplicate_chunks = []
        for chunk in chunks:
            if chunk.page_content not in chunk.metadata.values():
                chunk.page_content = f"{chunk.metadata.get('header 2', '')}\n{chunk.page_content}"
                no_duplicate_chunks.append(chunk)
        return no_duplicate_chunks

In [ ]:
embedder = ConfluenceChunkEmbedder()
chunker = ConfluenceChunker()

all_data = {}
total_embeddings = 0

for confluence_json_path in tqdm(os.listdir(SOURCE_PATH)):
    if confluence_json_path.endswith('.json'):

        with open (f'{SOURCE_PATH}/{confluence_json_path}', 'r', encoding='utf-8') as file:
            data = json.load(file)
            html_content = data['body']['view']['value']

        chunks = chunker.basic_chunk(html_content)
        if not chunks:
            continue
        embeddings = embedder.embed_chunks(chunks)

        page_metadata = {}
        page_metadata['page_link'] = data['_links']['webui']
        page_metadata['page_id'] = data['id']

        all_data[confluence_json_path] = {
            'chunks': chunks,
            'embeddings': embeddings,
            'page_metadata': page_metadata,
            'count': len(embeddings)
        }
        
        total_embeddings += len(embeddings)


print(f"✅ Создано {total_embeddings} эмбедингов")

with open('/content/drive/MyDrive/Colab Notebooks/confl-rag/embeddings_data_full.pkl', 'wb') as f:
    pickle.dump(all_data, f)
print("✅ Сохранено в Google Drive")